# LAB06 · Primeras consultas: de `head` a `SELECT`

## Bloque 2 · Sesión 4 · SQL con DuckDB


**Curso Big Data e IA Aplicada · Formación San Miguel**


---


Todo lo que hiciste en la terminal —contar, filtrar, agrupar— tiene un idioma universal que lo
expresa mejor. Hoy descubres que **ya lo sabías a medias**… y anclas la decisión más importante
del curso: **qué significa «limpio»**.


| Lo que traes | Lo que te llevas |
|---|---|
| `wc -l`, `cut`, `sort \| uniq -c`, `awk` | Las seis cláusulas de SQL |
| Números sueltos del Bloque 1 | Los **mismos** números, por segunda vía |
| «El fichero tiene suciedad» | **LIMPIO-v1**: la suciedad, definida por escrito y ejecutable |


---


### Las cuatro etiquetas — las mismas de siempre


| Etiqueta | A quién preguntas | Para qué |
|---|---|---|
| 🗂️ **RAG** | NotebookLM, con el material del curso | Conceptos. **Exígele que cite el apartado** |
| 🤖 **ASISTENTE** | Gemini o ChatGPT | Sintaxis, con el protocolo de cuatro pasos |
| ⚙️ **MÁQUINA** | A nadie: **se ejecuta** | Los números |
| 📝 **CRITERIO** | A nadie: **lo escribes tú** | Lo único que la IA no puede poner |


> **La regla:** si la respuesta es un número de tus datos, **no se le pregunta a ninguna IA. Se
> ejecuta.**


---


### ✍️ LAS TRES PREDICCIONES · antes de abrir nada


Escríbelas. No hace falta acertar: hace falta **mojarse**, porque una predicción escrita convierte
un número en un hallazgo.


**① ¿`COUNT(*)` sobre `ventas.csv` dirá 1.000.000 o 1.000.001? ¿Por qué?**


**② Los censos de suciedad del LAB02 (3030 · 2004 · 941 · 465), ¿saldrán idénticos en SQL?**


**③ Al limpiar el fichero, la facturación —429.888.864,70 €— ¿subirá o bajará?**


---

## Paso 1 · El motor · `BASE`


DuckDB vive **dentro de tu Python**: sin servidor, sin instalación de sistema, sin configuración.


> Si protesta con `ModuleNotFoundError`, es la lección del Bloque 1 sobre las instalaciones
> efímeras: `pip install duckdb` en la ▸ **Terminal** — y **reinicia el kernel**, o Python seguirá
> sin verlo.

In [ ]:
import duckdb

print("DuckDB", duckdb.__version__)

---


## Paso 2 · El `head` de SQL · `BASE`


Tu primera consulta. Fíjate en la ruta: lleva `../` porque **el cuaderno vive en `notebooks/` y los
datos un piso arriba**. Es la lección del LAB05, y sigue valiendo.


**La ruta ES la tabla.** No hay que importar nada.

In [ ]:
import duckdb

duckdb.sql("SELECT * FROM '../datasets/ventas.csv' LIMIT 5").show()

# Si falla con "No files found": te falta el ../ de la ruta.

---


## Paso 3 · El `wc -l` de SQL… con sorpresa · `BASE`


**¿Predicción ① escrita?** Entonces ejecuta.

In [ ]:
import duckdb

duckdb.sql("SELECT COUNT(*) AS filas FROM '../datasets/ventas.csv'").show()

# Esperado: 1000000

### 💡 Un millón justo — no 1.000.001


Tu `wc -l` contaba **líneas del fichero**, cabecera incluida. SQL entiende la primera línea como
**nombres de columna**, no como dato.


**Mismo fichero, dos niveles de abstracción.** Y tu primer número verificado por segunda vía
entre bloques.


✍️ **¿Acertaste la predicción ①? ¿Qué te hizo fallar, si fallaste?**


---


## Paso 4 · Los censos del LAB02, en UNA consulta · `BASE`


Lo que en el Bloque 1 fueron cuatro comandos separados, hoy es **una consulta con cuatro `FILTER`**.


`COUNT(*) FILTER (WHERE cond)` es un recuento condicionado **por columna**: te permite meter cuatro
censos en una sola pasada por el fichero.

In [ ]:
import duckdb

duckdb.sql("""SELECT
    COUNT(*) FILTER (WHERE COALESCE(TRIM(ciudad),'') = '') AS ciudades_vacias,
    COUNT(*) FILTER (WHERE ciudad = 'zaragoza')      AS zaragoza_minuscula,
    COUNT(*) FILTER (WHERE ciudad = ' Zaragoza')     AS zaragoza_espacio,
    COUNT(*) FILTER (WHERE precio_unitario <= 0)     AS precios_imposibles,
    COUNT(DISTINCT id_cliente)                       AS clientes_distintos
FROM '../datasets/ventas.csv'""").show()

# Esperado: 3030 · 2004 · 941 · 465 · 99996

✍️ **Compara cada cifra con tu bitácora del LAB02. ¿Alguna baila?**


---

### ⚠️ EL DETALLE QUE PARECE UNA ERRATA Y ES LA LECCIÓN DEL DÍA

**El manual escribe esa primera columna así:**

```sql
COUNT(*) FILTER (WHERE TRIM(ciudad) = '')     -- y dice "esperado 3030"
```

**Pruébalo en una celda nueva. Devuelve CERO.** No 3.030: **cero**.

✍️ **Antes de seguir leyendo: ¿por qué?**


> 💡 **Porque un campo vacío de un CSV no es una cadena vacía: DuckDB lo lee como `NULL`.** Y
> `TRIM(NULL)` es `NULL`, y `NULL = ''` no es «falso»: es **desconocido**. El `WHERE` solo deja pasar
> lo verdadero, así que no cuenta ni una.
>
> **Es la lógica de tres valores, y te la acabas de encontrar dos días antes de estudiarla.**

**Por eso el cuaderno usa `COALESCE(TRIM(ciudad), '') = ''`**, que es la forma que **no se puede
equivocar**: caza el `NULL`, caza la cadena vacía **y caza los espacios en blanco**. Compruébalo:

| Escrito así | `'  '` | `''` | `NULL` |
|---|---|---|---|
| `TRIM(ciudad) = ''` | ✅ | ✅ | ❌ |
| `ciudad IS NULL` | ❌ | ❌ | ✅ |
| **`COALESCE(TRIM(ciudad),'') = ''`** | ✅ | ✅ | ✅ |

> 🎯 **Y ahora la forma exacta del error, que es la del curso entero:** la consulta del manual **no
> falla**. No hay aviso, no hay rojo. **Devuelve un número plausible —cero— y sigue adelante.** Si
> nadie lo compara con el 3.030 del LAB02, ese fichero pasa por limpio.
>
> **Un método malo que hoy devuelve un número sigue siendo un método malo.**
>
> 🎯 **Dos herramientas, una verdad.** Comparar contra el Bloque 1 no es un trámite: es el método
> con el que se confía en un dato cuando nadie te regala la confianza. Lo repites mañana con Spark y
> en el proyecto con tres motores a la vez.
>
> 📌 Y guárdate los **99.996** de la última columna. Faltan cuatro clientes de cien mil. **Esa deuda
> se cobra dentro de dos horas**, y tiene nombres y apellidos.

---

## Paso 5 · `WHERE` con fechas · `BASE`

Las fechas son **fechas** (tipo `DATE`), no texto: se comparan de verdad.

In [ ]:
import duckdb

duckdb.sql("""SELECT COUNT(*) AS web_desde_nov
FROM '../datasets/ventas.csv'
WHERE canal = 'web' AND precio_unitario > 0 AND fecha >= DATE '2025-11-01'""").show()

# Esperado: 55553

---


## Paso 6 · El contador de frecuencias, con traje · `BASE`


Tu `sort | uniq -c | sort -rn` de toda la vida, dicho en una frase.

In [ ]:
import duckdb

duckdb.sql("""SELECT canal, COUNT(*) AS n
FROM '../datasets/ventas.csv'
WHERE precio_unitario > 0
GROUP BY canal
ORDER BY n DESC""").show()

# Esperado: tienda 499184 · web 333243 · movil 167108

✍️ **Apunta el reparto de los tres canales.** Spark tendrá que clavarlo en la sesión 6.


> 💡 **El secreto que evita el 80 % de los errores de novato:** el orden en que **escribes** no es
> el orden en que el motor **ejecuta**.
>
> ```
> FROM  ->  WHERE  ->  GROUP BY  ->  SELECT  ->  ORDER BY  ->  LIMIT
> ```
>
> Consecuencia práctica: un alias creado en el `SELECT` (`AS n`) **no existe todavía** cuando se
> evalúa el `WHERE`. Si necesitas filtrar por un agregado, la cláusula es `HAVING`, que corre
> después de agrupar. Cuando te salga el error, ya sabes por qué.


---

---


# ⚓ Paso 7 · LA CELDA-ANCLA · `BASE`


**El momento solemne del bloque.** Aquí deja de haber opiniones sobre qué es un dato limpio.


Léela cláusula a cláusula contra la definición del apartado 4.5 del manual:


| Regla de LIMPIO-v1 | Dónde vive en esta consulta |
|---|---|
| Fuera `precio_unitario <= 0` (las 465 imposibles) | el `WHERE` del final |
| Ciudad normalizada: espacios fuera, inicial en mayúscula | el `UPPER(SUBSTR(...))` con el `\|\|` |
| Ciudad vacía → `NULL`, **pero la fila se conserva** | el `CASE WHEN … THEN NULL` |


> El `CASE WHEN` es **el `if` de SQL**. La normalización de ciudad vive dentro de uno.
>
> ⚠️ **Por qué una vista y no otro fichero.** Podríamos guardar un `ventas_limpio.csv` y listo.
>
> Una **vista** —una consulta con nombre— no duplica datos, no puede desincronizarse del original
> (se recalcula al usarla) y **documenta la decisión en código ejecutable**: quien lee la vista lee
> la política de limpieza. El fichero físico llegará en el LAB10.
>
> **Definir primero, materializar después: ese orden ES el oficio.**

In [ ]:
import duckdb

duckdb.sql("""CREATE OR REPLACE VIEW ventas_limpio AS
SELECT id_venta, fecha, id_cliente, id_producto, categoria, unidades, precio_unitario,
       CASE WHEN COALESCE(TRIM(ciudad), '') = '' THEN NULL
            ELSE UPPER(SUBSTR(TRIM(ciudad),1,1))
                 || LOWER(SUBSTR(TRIM(ciudad),2)) END AS ciudad,
       canal
FROM '../datasets/ventas.csv'
WHERE precio_unitario > 0""")

duckdb.sql("SELECT COUNT(*) AS filas_limpias FROM ventas_limpio").show()

# Esperado: 999535   (= 1.000.000 - 465)
#
# El manual escribe aquí TRIM(ciudad)='' . Da el MISMO resultado -- pero por
# accidente: los vacíos ya llegan NULL y se propagan solos. Con COALESCE, la
# regla (3) del ancla hace de verdad lo que dice que hace.

### ✅ 999.535 = 1.000.000 − 465


✍️ **¿Te sale? Escribe el número aquí, que es el ancla del resto del curso:**


> ⚠️ **Si NO te sale 999.535, para y avisa.** Todo lo que viene después —el LAB07, el LAB08, el
> Spark del miércoles y la Fase 1 del proyecto— se compara contra este número. Un cuaderno con
> otro número no está «casi bien»: está mal.


---


## Paso 8 · La facturación de la verdad · `BASE`


**Predicción ③ a examen.** ¿Dijiste que subiría o que bajaría?

In [ ]:
import duckdb

duckdb.sql("""SELECT ROUND(SUM(unidades*precio_unitario), 2)            AS facturacion,
       ROUND(SUM(unidades*precio_unitario)/COUNT(*), 2)      AS ticket_medio
FROM ventas_limpio""").show()

# Esperado: 429892547.06 · 430.09

### 💡 ¡Limpiar **SUBIÓ** la facturación!


**+3.682,36 €** sobre la cifra sucia del Bloque 1 (429.888.864,70). ¿Por qué? Porque **los precios
negativos estaban restando**.


✍️ **¿Qué predijiste tú? ¿Y qué predijo la mayoría de la clase?**


> 🎯 La moraleja, a bitácora: **limpiar cambia resultados — por eso la definición se ancla por
> escrito.** Si dos analistas «correctos» limpian distinto, llevan dos verdades al comité. El ancla
> es lo que impide eso.
>
> 📌 **Los tres números que te llevas puestos de esta sesión:**
> **999.535 filas · 429.892.547,06 € · ticket 430,09 €.**
>
> A partir de aquí, **toda cifra del curso es sobre LIMPIO-v1 salvo aviso**. Ante cualquier número
> que no cuadre, tu primera pregunta es *«¿estoy sobre la vista limpia?»*.


---


## Paso 9 · Zaragoza recupera lo suyo · `COMPLETA`

In [ ]:
import duckdb

duckdb.sql("""SELECT COUNT(*) AS ventas
              FROM ventas_limpio WHERE ciudad = 'Zaragoza'""").show()

# Esperado: 340693   (las 2004 'zaragoza' + las 941 ' Zaragoza', de vuelta a casa)

> 💡 337.910 eran antes. Ahora **340.693**. Las 2.004 minúsculas y las 941 con espacio delante
> **han vuelto a casa** — y con ellas, el dinero que se habían llevado.


---


## Paso 10 · El mes pico · `RETO` · a casa


Sin código dado. Con `STRFTIME(fecha, '%Y-%m')` como llave, encuentra **qué mes facturó más**
sobre la vista limpia.


**Pista de control:** hay una carrera apretadísima en el podio. Nómbrala con números.


*(Celda libre debajo: `Esc` + `B`.)*


---

---


# 🏁 Qué te llevas de esta sesión


- Las **seis cláusulas** y su orden real de ejecución.
- `COUNT(*) FILTER`, `CASE WHEN`, fechas de verdad, `ROUND` y las vistas.
- Y sobre todo: **un ancla**. Una decisión de negocio fijada **por escrito y ejecutable**, que es
  exactamente lo que impide que dos análisis correctos den números distintos.


> **La frase de la sesión:** *sin ancla, no hay KPI: hay opiniones.*


**El cuaderno del LAB07 continúa aquí mismo** — y cobra la deuda de los cuatro clientes fantasma.

---
---

# LAB07 · JOINs: el pipeline se conecta
## Segunda mitad del día

Tres ficheros que vivían separados —**ventas**, **clientes**, **productos**— se convierten ahora en
un **modelo conectado**. Y de paso cobras una deuda que el curso tiene contigo desde la sesión 2.

```
clientes.csv   id_cliente(PK) · nombre · apellido · ciudad · segmento    100.000
ventas.csv     id_venta(PK) · id_cliente(FK) · id_producto(FK) · …     1.000.000
```

### El juguete que hay que aprenderse, y ahorra años de confusión

```
CLIENTES: (1,Ana) (2,Luis) (3,Marta)      VENTAS: (V1,cli 1) (V2,cli 1) (V3,cli 2)

INNER JOIN (solo parejas)   ->  Ana-V1 · Ana-V2 · Luis-V3
                                3 filas. Marta NO sale.
LEFT JOIN  (todo el izq.)   ->  Ana-V1 · Ana-V2 · Luis-V3 · Marta-NULL
                                4 filas. Marta sale CON NULL.
```

1. **Ana aparece dos veces** — el JOIN multiplica por cada pareja. **Cuenta el `COUNT` antes y
   después de todo JOIN nuevo.** Es el hábito nº 1 del oficio.

2. **El INNER hace desaparecer en silencio** a quien no tiene pareja.
3. **El LEFT la conserva con `NULL`… y ese `NULL` es información:** «cliente sin ventas».

El patrón `LEFT JOIN … WHERE derecha IS NULL` se llama **anti-join**, y hoy resuelve el misterio
del Bloque 1.

### ✍️ Dos predicciones más, rápidas

**① ¿Quiénes serán los 4 clientes sin ventas? Apuesta: ¿ciudad? ¿perfil?**


**② ¿El ticket medio de un cliente premium será mayor que el de un particular? Sí o no.**


---

## Paso 1 · El ancla sigue viva · `BASE`

**No hace falta recrearla:** la vista `ventas_limpio` vive en este mismo kernel desde el paso 7.
Una comprobación de tres segundos y seguimos.

> ⚠️ **Si has reiniciado el kernel**, vuelve a ejecutar la celda del ancla. Es la regla del curso:
> **cada cuaderno tiene que sobrevivir a un `Restart & Run All`.**

In [ ]:
import duckdb

duckdb.sql("SELECT COUNT(*) AS filas_limpias FROM ventas_limpio").show()

# Esperado: 999535.  Si falla, ejecuta otra vez la celda del ancla (paso 7).

---


## Paso 2 · Conocer la segunda tabla · `BASE`

In [ ]:
import duckdb

duckdb.sql("SELECT * FROM '../datasets/clientes.csv' LIMIT 3").show()

duckdb.sql("""SELECT segmento, COUNT(*) AS n
FROM '../datasets/clientes.csv'
GROUP BY segmento ORDER BY n DESC""").show()

# Esperado: particular 60116, y los otros dos 19951 y 19933.
# Suman 100.000 clientes exactos.

---


## Paso 3 · Tu primer JOIN · `BASE`


Cada venta conoce a su cliente. `USING (id_cliente)` sirve cuando la clave **se llama igual** en
ambas tablas; `ON v.id_cliente = c.id_cliente` es el caso general. Las letras `v` y `c` tras cada
nombre de tabla son **alias**: apodos que evitan escribir nombres kilométricos.

In [ ]:
import duckdb

duckdb.sql("""SELECT c.segmento,
       ROUND(SUM(v.unidades*v.precio_unitario)/1e6, 1) AS millones,
       COUNT(*)                                        AS operaciones
FROM ventas_limpio v
JOIN '../datasets/clientes.csv' c USING (id_cliente)
GROUP BY c.segmento
ORDER BY millones DESC""").show()

# Esperado: particular 258.7 M / 601073 ops · premium 85.7 / 199473
#           empresa 85.5 / 198989

### ✅ EL CONTROL DEL OFICIO — hazlo tú, en voz alta


```

601.073 + 199.473 + 198.989  =  999.535   ✓

```


✍️ **Suma tus tres recuentos de operaciones. ¿Dan exactamente las filas de tu vista limpia?**


> 🎯 **El JOIN no multiplicó ni perdió**: las claves eran sanas. Ese cuadre de diez segundos es el
> hábito nº 1 del oficio, y es lo que separa a quien une tablas de quien **cree** que las une.
>
> Cuando algún día no cuadre, ya sabrás dónde mirar: **una clave duplicada donde no debía**.


---


## Paso 4 · La hipótesis ③, a examen · `COMPLETA`

In [ ]:
import duckdb

duckdb.sql("""SELECT c.segmento, ROUND(AVG(v.unidades*v.precio_unitario), 2) AS ticket
FROM ventas_limpio v
JOIN '../datasets/clientes.csv' c USING (id_cliente)
GROUP BY c.segmento
ORDER BY ticket DESC""").show()

# Esperado: particular 430.48 · premium 429.58 · empresa 429.44  -> ¡empate técnico!

### 💡 La hipótesis razonable era **falsa**


El segmento cambia el **volumen** (cuántos compran), **no el ticket** (cuánto por compra).


✍️ **¿Qué habías predicho en la ③? ¿Con qué seguridad?**


> 🎯 Lección de humildad estadística, a bitácora: **una intuición de negocio puede ser mentira —
> por eso se mide antes de suponer.**
>
> *(Honestidad del material: nuestro generador reparte el ticket de forma uniforme; en datos reales
> suele haber diferencia. **Lo transferible es el método**, no este resultado concreto.)*


---


## Paso 5 · La tercera tabla, desde el JSON del LAB04 · `COMPLETA`


DuckDB lee el JSON anidado directamente y **reconoce la suma de stocks**: es **tu `jq` de la sesión
3, dicho en SQL**.

In [ ]:
import duckdb

duckdb.sql("""CREATE OR REPLACE VIEW productos AS
SELECT id, nombre, categoria, precio, stock.central + stock.tiendas AS stock_total
FROM read_json('../datasets/productos.json')""")

duckdb.sql("""
    SELECT p.nombre, ROUND(SUM(v.unidades*v.precio_unitario)/1e6, 2) AS millones
FROM ventas_limpio v
JOIN productos p ON v.id_producto = p.id
GROUP BY p.nombre
ORDER BY millones DESC
LIMIT 3""").show()

# Esperado: Monitor 27 QHD Compact 14.54 · Switch 8 puertos Compact 10.9
#           NAS 2 bahías i5 10.32

> 💡 Fíjate en `stock.central + stock.tiendas`. **Es el campo calculado que te inventaste con `jq`
> el otro día**, ahora dentro de una vista SQL. Otra herramienta, la misma idea: aplanar un árbol
> es siempre **decidir qué se conserva**.


---

---


# ⭐ Paso 6 · EL CLÍMAX: los cuatro fantasmas · `BASE`


**Tres sesiones lleva esperando esta salida.**


En el LAB02 contaste **99.996 clientes distintos** en un censo de 100.000, y el manual te dijo
«guárdate la pregunta». No era despiste: **era una promesa de diseño**. La respuesta necesitaba una
herramienta que aún no tenías.


El anti-join, sobre las tablas **completas** — los fantasmas no tienen ventas de ningún tipo, ni
siquiera sucias:

In [ ]:
import duckdb

duckdb.sql("""SELECT c.id_cliente, c.nombre, c.apellido, c.ciudad, c.segmento
FROM '../datasets/clientes.csv' c
LEFT JOIN '../datasets/ventas.csv' v USING (id_cliente)
WHERE v.id_venta IS NULL
ORDER BY c.id_cliente""").show()

# Esperado, cuatro filas:
#    1373  Ivan Jimenez     Zaragoza   empresa
#   28460  Emma Iglesias    Teruel     particular
#   55011  Hugo Rubio       Barcelona  premium
#   93208  Mario Serrano    Zaragoza   particular

### 💡 Cuatro personas con nombre


Un recuento se ha convertido en **una historia de negocio**. Y la mejor pregunta la hace uno de
ellos solito: **¿por qué un cliente premium no ha comprado jamás?** Eso vale una llamada comercial.


✍️ **¿Acertaste algo de la predicción ②? ¿Y qué harías con esta lista si trabajaras ahí?**


> ⚠️ **Si te sale vacío:** has escrito `= NULL`. Vuelve a la lógica de tres valores: en SQL una
> comparación puede ser **desconocida**, y `NULL = NULL` también lo es. **Nada que toque un `NULL`
> pasa jamás un `WHERE`.** La única forma de preguntar es `IS NULL`.
>
> 📌 **Anota los cuatro nombres.** Son el **indicador ④** de la Fase 2 del proyecto, y ahí se piden
> **por nombre**.


---


## Paso 7 · El premium fantasma · `RETO` · a casa


De los 19.933 premium, ¿cuántos no compraron jamás? Escribe la consulta —**la tienes casi hecha**—
y ponle nombre y apellido al resultado.


**Control:** la respuesta cabe en una fila.


---

---


# 🔍 CONSULTA · Bloque N


---


**N1 · 🗂️ RAG · `BASE`**


> *Según el manual, ¿qué es un anti-join y por qué el `INNER JOIN` no sirve para encontrar a los
> clientes sin ventas? Cita el apartado.*


✍️


---


**N2 · ⚙️ MÁQUINA · `BASE`**


✍️ **Los cuatro fantasmas: id, nombre, ciudad y segmento de cada uno.**


---


**N3 · 📝 CRITERIO · `BASE`**


✍️ **Explica con tus palabras por qué `COUNT(*)` y `COUNT(ciudad)` pueden dar números distintos
sobre la misma tabla — y qué implica eso para un informe «por ciudad».**


---


**N4 · ⚙️ MÁQUINA · `BASE`**


✍️ **El control del oficio: la suma de operaciones de los tres segmentos, y las filas de tu vista
limpia. ¿Cuadra?**


---


**N5 · 🤖 ASISTENTE · `COMPLETA`**


> *Tengo `ventas_limpio` y `clientes.csv`. Escríbeme la consulta que devuelve los clientes SIN
> ninguna venta, y explícame pieza a pieza por qué usas ese tipo de JOIN.*


**Compárala con la tuya. ¿Usó `IS NULL` o cayó en el `= NULL`?**


✍️


---


**N6 · 📝 CRITERIO · `COMPLETA`**


✍️ **La hipótesis del ticket premium era falsa. ¿Qué habrías hecho si hubieras entregado un informe
basado en esa intuición, sin medirla?**


---

# 🔍 CONSULTA · Bloque S

**Cuatro preguntas, una de cada etiqueta.** Menos que otras veces y a propósito: se contestan
**en clase**, no en casa.

---

**S1 · 🗂️ RAG**

> *Según el manual, ¿por qué se dice que SQL es «declarativo» y en qué se diferencia del pipeline
> que montaste en el LAB03? **Cítame el apartado exacto.** Si no está en tus fuentes, dímelo.*

✍️


---

**S2 · ⚙️ MÁQUINA** *(esto no se le pregunta a nadie: se ejecuta)*

✍️ **Tus tres números del ancla — filas, facturación, ticket — y los cuatro fantasmas con nombre.**


---

**S3 · 🤖 ASISTENTE**

> *Tengo una tabla `ventas_limpio` y un `clientes.csv`. Escríbeme la consulta que devuelve los
> clientes SIN ninguna venta, y explícame pieza a pieza por qué usas ese tipo de JOIN.*

**Audítala contra la tuya: ¿usó `IS NULL` o cayó en el `= NULL`?** Si cayó, ejecútalo y enseña que
devuelve vacío **sin dar ningún error**. Ese es el error silencioso del día.

✍️


---

**S4 · 📝 CRITERIO** *(lo único que la IA no puede poner)*

✍️ **LIMPIO-v1 conserva las filas con ciudad vacía en vez de borrarlas. ¿Por qué es una decisión y
no un descuido? ¿Qué se perdería borrándolas?**


---
---

# 📦 Entregable del día

**`Ctrl+S` antes de archivar.** La celda copia el fichero **guardado en disco**, no lo que ves en
pantalla.

| # | Lo que tiene que estar | ¿Hecho? |
|---|---|---|
| 1 | **El ancla**: 999.535 · 429.892.547,06 € · ticket 430,09 | |
| 2 | Las **predicciones** escritas *antes*, y puntuadas | |
| 3 | El **cuadre del JOIN**: 601.073 + 199.473 + 198.989 = 999.535 | |
| 4 | **Los cuatro fantasmas**, con nombre, ciudad y segmento | |
| 5 | El **bloque S** contestado | |

In [ ]:
import shutil, os, glob, json

SESION = 7                      # numeracion del MANUAL (sesiones 4, 5 y 6)

# ══════════════════════════════════════════════════════════════════════════
#  GUARDIÁN · ¿está en el disco lo que ves en pantalla?
#
#  Esta celda copia el FICHERO DEL DISCO, no lo que tienes delante. Jupyter
#  guarda solo cada pocos minutos: si archivas antes de un Ctrl+S, entregas
#  el cuaderno SIN tus resultados y el HTML sale vacío.
# ══════════════════════════════════════════════════════════════════════════

def resultados_en_disco(ruta):
    try:
        nb = json.load(open(ruta, encoding="utf-8"))
    except Exception:
        return 0, 0
    codigo = [c for c in nb["cells"] if c["cell_type"] == "code"]
    return sum(1 for c in codigo if c.get("outputs")), len(codigo)

cuadernos = [f for f in glob.glob("*lab06*.ipynb") if ".ipynb_checkpoints" not in f]
listo = bool(cuadernos)

if not cuadernos:
    print("  No encuentro los cuadernos de esta sesión en esta carpeta.")

for cuaderno in cuadernos:
    hechas, total = resultados_en_disco(cuaderno)
    print(f"  {cuaderno}: {hechas} de {total} celdas con resultados en el disco")
    if hechas == 0:
        listo = False

if not listo:
    print()
    print("  " + "=" * 68)
    print("   PARA AQUI. No he archivado nada.")
    print("   Pulsa  Ctrl+S  y vuelve a ejecutar ESTA celda.")
    print("  " + "=" * 68)
else:
    os.makedirs("entregables", exist_ok=True)
    piezas = cuadernos + [f for f in ["mi_bitacora.ipynb"] if os.path.exists(f)]
    for pieza in piezas:
        destino = f"entregables/S{SESION:02d}_{os.path.basename(pieza)}"
        shutil.copy(pieza, destino)
        print("  copiado:", destino)

In [ ]:
import glob, os

# Exporta a HTML. El HTML conserva las salidas incrustadas: se manda por
# correo y se ve entero, sin entorno, sin kernel, sin nada.
for cuaderno in glob.glob("*lab06*.ipynb"):
    if ".ipynb_checkpoints" in cuaderno:
        continue
    salida = f"S{SESION:02d}_" + os.path.splitext(os.path.basename(cuaderno))[0]
    !jupyter nbconvert --to html --output-dir entregables --output {salida} "{cuaderno}"

In [ ]:
import glob, os, re

# ══════════════════════════════════════════════════════════════════════════
#  LA COMPROBACIÓN QUE CIERRA EL CÍRCULO
#
#  Un cuaderno ejecutado deja en el HTML el número de cada celda: [1]:, [2]:...
#  Si no hay ninguno, el HTML NO lleva tus resultados.
#  El TAMAÑO del fichero engaña; este número, no.
# ══════════════════════════════════════════════════════════════════════════

def celdas_ejecutadas(ruta_html):
    h = open(ruta_html, encoding="utf-8", errors="ignore").read()
    return len(re.findall(r"\[[0-9]+\]:", h)), h.count("data:image/png;base64")

htmls = sorted(glob.glob("entregables/*.html"))
vacios = []

if not htmls:
    print("  No hay ningún HTML. ¿Ejecutaste la celda anterior?")

for h in htmls:
    ejecutadas, graficas = celdas_ejecutadas(h)
    kb = os.path.getsize(h) / 1024
    if ejecutadas == 0:
        vacios.append(os.path.basename(h))
    estado = "OK    " if ejecutadas else "VACIO "
    print(f"  {estado} {os.path.basename(h):<34}"
          f" {ejecutadas:>3} celdas ejecutadas · {kb:.0f} KB")

print()
if vacios:
    print("  Estos HTML no llevan resultados:", ", ".join(vacios))
    print("  Ctrl+S y repite las dos celdas anteriores.")
elif htmls:
    print("  Todo correcto. Ya puedes empaquetar.")